# EngressNet Data Preparation: Perfect Model

Notebook front-end for the `sea_ice_downscaling` package. Each step below calls the same functions `build_dataset.run_pipeline()` calls internally, but broken out so you can inspect `X_ds` / `Y_ds`, plot intermediate results, and re-run individual steps without rebuilding everything.

**Before running:** Put the `functions/` package folder in the same directory as this notebook (or anywhere on your `PYTHONPATH`), and make sure the packages in `requirements.txt` are installed in this kernel's environment.

## 1. Setup

In [1]:
import torch
import xarray as xr
import matplotlib.pyplot as plt

from functions.config import PipelineConfig, REGIONS, VAR_COMPONENT
from functions.file_discovery import collect_files, discover_member_dirs, summarize_collection
from functions.grids import build_grid_bundle
from functions.dataset_builder import build_both_predictor_datasets, build_target_dataset
from functions.io_utils import save_dataset, compute_and_save_scaling

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cpu


## 2. Configure the run

Edit the values below, then re-run this cell whenever you want a different region, destination grid, or predictor pipeline.

Available regions: `list(REGIONS)` (currently `cambridge_bay`, `kivalina`). Add new ones in `functions/config.py`.

`lr_dest_grid` controls where the **coarsened ice/atm predictor channels** (`X`) land:
- `"1deg"` -- legacy rectilinear destination (original script's behavior)
- `"0p1deg"` -- finer rectilinear destination
- `"ease2_n25km"` -- NSIDC EASE-Grid 2.0 Northern Hemisphere, 25 km (only valid with `use_area_average_lr=False`)

`use_area_average_lr` picks which of the two predictor pipelines builds `X`:
- `False` (default) -- **interpolated xESMF pipeline**: regrid HR-native straight to `lr_dest_grid` with xESMF (bilinear for ice, nearest-neighbor for atm).
- `True` -- **native area-mean pipeline**: regrid HR-native to `hr_dest_grid` with xESMF, then cos-latitude-weighted block-average down to `lr_dest_grid`. Requires `lr_dest_grid` to be rectilinear (`"1deg"` or `"0p1deg"`) -- raises at config-construction time if combined with `"ease2_n25km"`.

In [2]:
print("Available regions:", list(REGIONS), "\n")

config = PipelineConfig(
    region=REGIONS["kivalina"],   # swap for REGIONS["kivalina"], etc.
    lr_dest_grid="1deg",               # "1deg" / "0p1deg" / "ease2_n25km" (xESMF pipeline only)
    use_area_average_lr=False,         # True = native area-mean pipeline instead of interpolated xESMF
    start_year=1920,
    output_dir="/glade/derecho/scratch/skygale/Downscaling_Data/",
)

config

Available regions: ['cambridge_bay', 'kivalina'] 



PipelineConfig(low_res_glob='/glade/campaign/collections/gdex/data/d651030/BHIST/*', high_res_glob='/glade/campaign/collections/gdex/data/d651007/b.e13.*', low_vars=('hi', 'aice', 'U', 'V'), target_var='hi', start_year=1920, use_area_average_lr=False, region=RegionBBox(name='kivalina', lon_min=-190, lon_max=-140, lat_min=60, lat_max=80), lr_dest_grid='1deg', hr_dest_grid='0p1deg', grid_paths=GridPaths(atm_scrip_lr='/glade/p/cesmdata/cseg/inputdata/share/scripgrids/ne30np4_091226_pentagons.nc', atm_scrip_hr='/glade/p/cesmdata/cseg/inputdata/share/scripgrids/ne120np4_pentagons_100310.nc', pop_grid_lr='POP_gx1v7', pop_grid_hr='POP_tx0.1v2'), output_dir='/glade/derecho/scratch/skygale/Downscaling_Data/', created_by='Sky Gale', weights_dir='/glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights', max_workers_io=64, max_workers_hr=2)

## 3. Discover ensemble member directories and history files

`summarize_collection` prints per-member file counts and flags any member/variable combination with zero files.

In [3]:
low_res_dirs = discover_member_dirs(config.low_res_glob)
high_res_dirs = discover_member_dirs(config.high_res_glob)

print(f"{len(low_res_dirs)} low-res members, {len(high_res_dirs)} high-res members")

10 low-res members, 10 high-res members


In [4]:
low_res_files = collect_files(low_res_dirs, config.low_vars, VAR_COMPONENT, config.start_year)
high_res_files = collect_files(high_res_dirs, [config.target_var], VAR_COMPONENT, config.start_year)
coarsen_files = collect_files(high_res_dirs, config.low_vars, VAR_COMPONENT, config.start_year)

summarize_collection("Low-res ", low_res_files)
summarize_collection("High-res", high_res_files)
summarize_collection("Coarsen ", coarsen_files)

Low-res  | # ens: 10 | # vars: 4
High-res | # ens: 10 | # vars: 1
Coarsen  | # ens: 10 | # vars: 4


## 4. Build native grids and regridders

This builds the POP/SCRIP source grids, the destination grid(s), and all `xe.Regridder` objects (cached to `config.weights_dir` and reused on subsequent runs unless you change a grid definition).

In [5]:
import warnings
warnings.filterwarnings("ignore", message="Latitude is outside of \\[-90, 90\\]")

In [6]:
grids = build_grid_bundle(config)
print("Regridders built:", list(grids.regridders))

Regridders built (reused from cache where the weight file already existed):
  ice_hr_to_lr: /glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights/ice_hr_to_1deg.nc
  ice_hr_to_hr: /glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights/ice_hr_to_0p1deg.nc
  atm_hr_to_lr: /glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights/atm_hr_to_1deg.nc
  atm_hr_to_hr: /glade/work/skygale/_projects/SeaIceDownscaling/regrid_weights/atm_hr_to_0p1deg.nc
Regridders built: ['ice_hr_to_lr', 'ice_hr_to_hr', 'atm_hr_to_lr', 'atm_hr_to_hr']


## 5. Build X (low-res predictors, coarsened from high-res)

We run both predictor pipelines from the **same HR source files** (`coarsen_files`) and the **same `grids` bundle**:

| variable | pipeline | what happens |
|---|---|---|
| `X_interp_ds` | interpolated xESMF | HR native → `ice_hr_to_lr` / `atm_hr_to_lr` regridder directly to `lr_dest_grid` |
| `X_area_ds` | native area-mean | HR native → `ice_hr_to_hr` / `atm_hr_to_hr` regridder to `hr_dest_grid`, then cos-lat-weighted block average → `lr_dest_grid` |

Keeping both lets you swap `X_interp_ds` vs `X_area_ds` into the same model training run and directly compare the effect of the coarsening method on skill, without rebuilding everything from scratch.

> **Note:** the area-mean path requires `lr_dest_grid` to be rectilinear (`'1deg'` or `'0p1deg'`). If you switch to `lr_dest_grid='ease2_n25km'` in the config cell, only `X_interp_ds` will be valid; `X_area_ds` will raise at construction time (caught below).

In [7]:
X_interp_ds, X_area_ds = build_both_predictor_datasets(coarsen_files, grids, config)
print("X_interp dims:", dict(X_interp_ds.sizes))
print("X_areadims:", dict(X_area_ds.sizes))

Processing 360 files (10 members × 4 vars, both pipelines)...


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [ ]:
ncols = 2 if _area_available else 1
fig, axes = plt.subplots(1, ncols, figsize=(6 * ncols, 5))
if ncols == 1:
    axes = [axes]

X_interp_ds["X"].isel(ensemble=0, time=0, channel=0).plot(ax=axes[0])
axes[0].set_title(f"X_interp ch=0 ({config.low_vars[0]}), ens=0, t=0\n(xESMF interpolated → {config.lr_dest_grid})")

if _area_available:
    X_area_ds["X"].isel(ensemble=0, time=0, channel=0).plot(ax=axes[1])
    axes[1].set_title(f"X_area ch=0 ({config.low_vars[0]}), ens=0, t=0\n(HR regrid → cos-lat avg → {config.lr_dest_grid})")

plt.tight_layout()
plt.show()

## 6. Build Y (high-res target)


In [ ]:
Y_ds = build_target_dataset(high_res_files, grids, config)
Y_ds

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
Y_ds["Y"].isel(ensemble=0, time=0, channel=0).plot(ax=ax)
ax.set_title(f"Y ({config.target_var}), ensemble 0, t=0")
plt.show()

## 7. Save X and Y

Defaults to chunked Zarr (`fmt="zarr"`), which is what NCAR MILES's CREDIT platform expects.
Pass `fmt="netcdf"` instead if you need a `.nc` file for something else.

`compute_and_save_scaling` writes a small `*_scaling.nc` sidecar with per-channel mean/std --
compute this from training-split data only once you have a train/val/test split; right now it's
computed over everything you built above.


In [ ]:
x_path = save_dataset(X_ds, config.output_dir, "X_perfmodexp_cons", fmt="zarr")
compute_and_save_scaling(X_ds, "X", config.output_dir, "X_perfmodexp_cons")

In [ ]:
y_path = save_dataset(Y_ds, config.output_dir, "Y_perfmodexp", fmt="zarr")
compute_and_save_scaling(Y_ds, "Y", config.output_dir, "Y_perfmodexp")

print("Done.")
print("  X:", x_path)
print("  Y:", y_path)

## Appendix: run everything in one call

If you ever just want steps 3-7 above run end-to-end without inspecting anything in between
(e.g. for a final production run once you trust the config), `run_pipeline` does exactly that:


In [ ]:
# from sea_ice_downscaling.build_dataset import run_pipeline
# run_pipeline(config, save_fmt="zarr")